In [5]:
! pip install sec_edgar_api

In [6]:
import requests

url = "https://www.sec.gov/files/company_tickers.json"
headers = {"User-Agent": "prakharlearndata@gmail.com"}
data = requests.get(url, headers=headers).json()

for record in data.values():
    if record["ticker"] == "AAPL":  # Change ticker here
        cik = str(record["cik_str"]).zfill(10)  # pad with zeros
        print(cik)


0000320193


In [7]:
from sec_edgar_api import EdgarClient
edgar = EdgarClient(user_agent="prakharlearndata@gmail.com")
edgar.get_submissions(cik="0000320193")


{'cik': '0000320193',
 'entityType': 'operating',
 'sic': '3571',
 'sicDescription': 'Electronic Computers',
 'ownerOrg': '06 Technology',
 'insiderTransactionForOwnerExists': 0,
 'insiderTransactionForIssuerExists': 1,
 'name': 'Apple Inc.',
 'tickers': ['AAPL'],
 'exchanges': ['Nasdaq'],
 'ein': '942404110',
 'lei': None,
 'description': '',
 'website': '',
 'investorWebsite': '',
 'category': 'Large accelerated filer',
 'fiscalYearEnd': '0927',
 'stateOfIncorporation': 'CA',
 'stateOfIncorporationDescription': 'CA',
 'addresses': {'mailing': {'street1': 'ONE APPLE PARK WAY',
   'street2': None,
   'city': 'CUPERTINO',
   'stateOrCountry': 'CA',
   'zipCode': '95014',
   'stateOrCountryDescription': 'CA',
   'isForeignLocation': 0,
   'foreignStateTerritory': None,
   'country': None,
   'countryCode': None},
  'business': {'street1': 'ONE APPLE PARK WAY',
   'street2': None,
   'city': 'CUPERTINO',
   'stateOrCountry': 'CA',
   'zipCode': '95014',
   'stateOrCountryDescription': 'CA

In [8]:
import re
import requests
import pandas as pd

def get_10k_htm_links(ticker):
    headers = {"User-Agent": "Your Name your.email@example.com"}
    
    # Step 1 — Get CIK for ticker
    tickers_url = "https://www.sec.gov/files/company_tickers.json"
    resp = requests.get(tickers_url, headers=headers)
    resp.raise_for_status()
    data = resp.json()

    cik = None
    for record in data.values():
        if record["ticker"].lower() == ticker.lower():
            cik = str(record["cik_str"]).zfill(10)
            cik_int = str(record["cik_str"])  # integer form for URLs
            break
    if not cik:
        raise ValueError(f"CIK not found for ticker {ticker}")

    # Step 2 — Get submissions JSON
    submissions_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    resp = requests.get(submissions_url, headers=headers)
    resp.raise_for_status()
    sub_data = resp.json()

    # Step 3 — Loop over 10-K filings
    recent_filings = sub_data["filings"]["recent"]
    form_types = recent_filings["form"]
    accession_numbers = recent_filings["accessionNumber"]
    filing_dates = recent_filings["filingDate"]

    results = []
    pattern = re.compile(rf"^{ticker.lower()}-\d{{8}}\.htm$")

    for i, form in enumerate(form_types):
        if form == "10-K":
            acc_no_no_dashes = accession_numbers[i].replace("-", "")
            
            # Get filing index.json
            filing_index_url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_no_no_dashes}/index.json"
            filing_resp = requests.get(filing_index_url, headers=headers)
            if filing_resp.status_code != 200:
                continue
            filing_data = filing_resp.json()
            
            # Filter for pattern-matching HTML files
            for file_info in filing_data.get("directory", {}).get("item", []):
                filename = file_info["name"]
                if pattern.match(filename):
                    htm_url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_no_no_dashes}/{filename}"
                    results.append({
                        "filing_date": filing_dates[i],
                        "cik": cik,
                        "ticker": ticker.upper(),
                        "url": htm_url
                    })

    # Step 4 — Convert to DataFrame
    df = pd.DataFrame(results)
    return df

# Example usage for Apple
df_aapl = get_10k_htm_links("AMZN")
print(df_aapl)


  filing_date         cik ticker  \
0  2025-02-07  0001018724   AMZN   
1  2024-02-02  0001018724   AMZN   
2  2023-02-03  0001018724   AMZN   
3  2022-02-04  0001018724   AMZN   
4  2021-02-03  0001018724   AMZN   

                                                 url  
0  https://www.sec.gov/Archives/edgar/data/101872...  
1  https://www.sec.gov/Archives/edgar/data/101872...  
2  https://www.sec.gov/Archives/edgar/data/101872...  
3  https://www.sec.gov/Archives/edgar/data/101872...  
4  https://www.sec.gov/Archives/edgar/data/101872...  


In [9]:
df_aapl

,filing_date,cik,ticker,url
0,2025-02-07,0001018724,AMZN,https://www.sec.gov/Archives/edgar/data/101872...
1,2024-02-02,0001018724,AMZN,https://www.sec.gov/Archives/edgar/data/101872...
2,2023-02-03,0001018724,AMZN,https://www.sec.gov/Archives/edgar/data/101872...
3,2022-02-04,0001018724,AMZN,https://www.sec.gov/Archives/edgar/data/101872...
4,2021-02-03,0001018724,AMZN,https://www.sec.gov/Archives/edgar/data/101872...


In [10]:
import os
import re
import requests
import pandas as pd
from datetime import datetime

class SEC10KFetcher:
    BASE_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
    BASE_SUBMISSION_URL = "https://data.sec.gov/submissions/CIK{}.json"
    BASE_FILING_INDEX_URL = "https://www.sec.gov/Archives/edgar/data/{}/{}/index.json"
    BASE_ARCHIVES_URL = "https://www.sec.gov/Archives/edgar/data/{}/{}/{}"

    def __init__(self, user_agent, download_dir="download"):
        self.headers = {"User-Agent": user_agent}
        self.download_dir = download_dir
        os.makedirs(download_dir, exist_ok=True)
        self.ticker_cik_map = self._load_ticker_cik_map()

    def _load_ticker_cik_map(self):
        """Load mapping of ticker to CIK."""
        resp = requests.get(self.BASE_TICKERS_URL, headers=self.headers)
        resp.raise_for_status()
        data = resp.json()
        mapping = {}
        for record in data.values():
            mapping[record["ticker"].lower()] = {
                "cik_str": str(record["cik_str"]),
                "cik_padded": str(record["cik_str"]).zfill(10)
            }
        return mapping

    def _get_cik_info(self, ticker):
        """Retrieve CIK details for a ticker."""
        return self.ticker_cik_map.get(ticker.lower())

    def get_10k_htm_links(self, ticker):
        cik_info = self._get_cik_info(ticker)
        if not cik_info:
            raise ValueError(f"CIK not found for ticker {ticker}")

        cik_padded = cik_info["cik_padded"]
        cik_str = cik_info["cik_str"]

        # Get submissions JSON
        submissions_url = self.BASE_SUBMISSION_URL.format(cik_padded)
        resp = requests.get(submissions_url, headers=self.headers)
        resp.raise_for_status()
        sub_data = resp.json()

        recent_filings = sub_data["filings"]["recent"]
        form_types = recent_filings["form"]
        accession_numbers = recent_filings["accessionNumber"]
        filing_dates = recent_filings["filingDate"]

        results = []
        pattern = re.compile(rf"^{ticker.lower()}-\d{{8}}\.htm$")

        for i, form in enumerate(form_types):
            if form == "10-K":
                acc_no_no_dashes = accession_numbers[i].replace("-", "")
                filing_index_url = self.BASE_FILING_INDEX_URL.format(cik_str, acc_no_no_dashes)
                filing_resp = requests.get(filing_index_url, headers=self.headers)
                if filing_resp.status_code != 200:
                    continue
                filing_data = filing_resp.json()

                for file_info in filing_data.get("directory", {}).get("item", []):
                    filename = file_info["name"]
                    if pattern.match(filename):
                        htm_url = self.BASE_ARCHIVES_URL.format(cik_str, acc_no_no_dashes, filename)
                        results.append({
                            "filing_date": filing_dates[i],
                            "cik": cik_padded,
                            "ticker": ticker.upper(),
                            "url": htm_url
                        })

        df = pd.DataFrame(results)
        self._save_csv(ticker, df)
        return df

    def _save_csv(self, ticker, df):
        """Save DataFrame to timestamped and latest CSVs."""
        ticker_dir = os.path.join(self.download_dir, ticker.upper())
        os.makedirs(ticker_dir, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_filename = f"{ticker.upper()}_{timestamp}.csv"
        csv_path = os.path.join(ticker_dir, csv_filename)
        df.to_csv(csv_path, index=False)

        latest_dir = os.path.join(ticker_dir, "latest")
        os.makedirs(latest_dir, exist_ok=True)
        latest_csv_path = os.path.join(latest_dir, "latest.csv")
        df.to_csv(latest_csv_path, index=False)

        print(f"✅ Saved timestamped CSV: {csv_path}")
        print(f"✅ Saved latest CSV: {latest_csv_path}")

    def get_multiple_tickers(self, tickers):
        """Fetch 10-K HTML links for multiple tickers."""
        all_results = []
        for ticker in tickers:
            try:
                df = self.get_10k_htm_links(ticker)
                all_results.append(df)
            except Exception as e:
                print(f"⚠️ Error processing {ticker}: {e}")
        if all_results:
            return pd.concat(all_results, ignore_index=True)
        return pd.DataFrame()


# ---------------- USAGE EXAMPLE ----------------
if __name__ == "__main__":
    fetcher = SEC10KFetcher(user_agent="Your Name your.email@example.com")
    tickers = ["AMZN", "AAPL"]
    df_all = fetcher.get_multiple_tickers(tickers)
    print(df_all)


✅ Saved timestamped CSV: download\AMZN\AMZN_20250811_123444.csv
✅ Saved latest CSV: download\AMZN\latest\latest.csv
✅ Saved timestamped CSV: download\AAPL\AAPL_20250811_123449.csv
✅ Saved latest CSV: download\AAPL\latest\latest.csv
  filing_date         cik ticker  \
0  2025-02-07  0001018724   AMZN   
1  2024-02-02  0001018724   AMZN   
2  2023-02-03  0001018724   AMZN   
3  2022-02-04  0001018724   AMZN   
4  2021-02-03  0001018724   AMZN   
5  2024-11-01  0000320193   AAPL   
6  2023-11-03  0000320193   AAPL   
7  2022-10-28  0000320193   AAPL   
8  2021-10-29  0000320193   AAPL   
9  2020-10-30  0000320193   AAPL   

                                                 url  
0  https://www.sec.gov/Archives/edgar/data/101872...  
1  https://www.sec.gov/Archives/edgar/data/101872...  
2  https://www.sec.gov/Archives/edgar/data/101872...  
3  https://www.sec.gov/Archives/edgar/data/101872...  
4  https://www.sec.gov/Archives/edgar/data/101872...  
5  https://www.sec.gov/Archives/edgar/dat

In [11]:
import os
import re
import requests
import pandas as pd
from datetime import datetime

class Sec10KFetcher:
    def __init__(self, tickers, base_dir="download"):
        self.tickers = [t.upper() for t in tickers]
        self.base_dir = base_dir
        self.headers = {"User-Agent": "Your Name your.email@example.com"}
        os.makedirs(os.path.join(self.base_dir, "file"), exist_ok=True)
        self.ticker_cik_map = self._load_ticker_cik_mapping()

    def _load_ticker_cik_mapping(self):
        """Download ticker-to-CIK mapping from SEC."""
        tickers_url = "https://www.sec.gov/files/company_tickers.json"
        resp = requests.get(tickers_url, headers=self.headers)
        resp.raise_for_status()
        data = resp.json()
        mapping = {record["ticker"].upper(): str(record["cik_str"]).zfill(10)
                   for record in data.values()}
        return mapping

    def _get_10k_links_for_ticker(self, ticker):
        """Fetch all matching 10-K HTM links for a single ticker."""
        cik = self.ticker_cik_map.get(ticker)
        if not cik:
            print(f"⚠ CIK not found for ticker {ticker}")
            return []

        cik_int = str(int(cik))  # integer form for URLs
        submissions_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
        resp = requests.get(submissions_url, headers=self.headers)
        if resp.status_code == 404:
            print(f"⚠ No submissions found for {ticker}")
            return []
        resp.raise_for_status()
        sub_data = resp.json()

        results = []
        recent_filings = sub_data.get("filings", {}).get("recent", {})
        form_types = recent_filings.get("form", [])
        accession_numbers = recent_filings.get("accessionNumber", [])
        filing_dates = recent_filings.get("filingDate", [])

        pattern = re.compile(rf"^{ticker.lower()}-\d{{8}}\.htm$")

        for i, form in enumerate(form_types):
            if form == "10-K":
                acc_no_no_dashes = accession_numbers[i].replace("-", "")
                filing_index_url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_no_no_dashes}/index.json"
                filing_resp = requests.get(filing_index_url, headers=self.headers)
                if filing_resp.status_code != 200:
                    continue
                filing_data = filing_resp.json()

                for file_info in filing_data.get("directory", {}).get("item", []):
                    filename = file_info["name"]
                    if pattern.match(filename):
                        htm_url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_no_no_dashes}/{filename}"
                        results.append({
                            "filing_date": filing_dates[i],
                            "cik": cik,
                            "ticker": ticker,
                            "url": htm_url
                        })
        return results

    def fetch_all(self):
        """Fetch data for all tickers and save combined CSV."""
        all_results = []
        for ticker in self.tickers:
            print(f"Fetching 10-K HTM links for {ticker}...")
            ticker_results = self._get_10k_links_for_ticker(ticker)
            all_results.extend(ticker_results)

        df = pd.DataFrame(all_results)
        if df.empty:
            print("⚠ No data found for any ticker.")
            return df

        # Save combined CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        timestamped_csv = os.path.join(self.base_dir, "file", f"{timestamp}.csv")
        latest_csv = os.path.join(self.base_dir, "file", "latest.csv")

        df.to_csv(timestamped_csv, index=False)
        df.to_csv(latest_csv, index=False)

        print(f"✅ Saved timestamped CSV: {timestamped_csv}")
        print(f"✅ Saved latest CSV: {latest_csv}")
        return df


# === Example Usage ===
if __name__ == "__main__":
    tickers = ["AAPL", "AMZN", "MSFT"]
    fetcher = Sec10KFetcher(tickers)
    df_all = fetcher.fetch_all()
    print(df_all)


KeyboardInterrupt: 

In [ ]:
import os
import pandas as pd
import requests

class SECFilingsDownloader:
    def __init__(self, base_dir="download", user_agent="Your Name Contact@domain.com"):
        self.base_dir = base_dir
        self.headers = {
            "User-Agent": user_agent,
            "Accept-Encoding": "gzip, deflate",
            "Host": "www.sec.gov"
        }
        os.makedirs(self.base_dir, exist_ok=True)

    def download_for_ticker(self, ticker):
        """Download all files listed in the latest.csv for a given ticker."""
        latest_csv_path = os.path.join(self.base_dir,  "file", "latest.csv")

        if not os.path.exists(latest_csv_path):
            print(f"[ERROR] latest.csv not found  at {latest_csv_path}")
            return

        df = pd.read_csv(latest_csv_path)
        ticker_folder = os.path.join(self.base_dir, ticker)
        os.makedirs(ticker_folder, exist_ok=True)

        for _, row in df.iterrows():
            url = row.get('url')
            if not url:
                continue

            filename = os.path.basename(url)
            filepath = os.path.join(ticker_folder, filename)

            if os.path.exists(filepath):
                print(f"[SKIP] {ticker}: {filename} already exists.")
                continue

            try:
                response = requests.get(url, headers=self.headers)
                response.raise_for_status()
                with open(filepath, "wb") as f:
                    f.write(response.content)
                print(f"[OK] {ticker}: {filename} downloaded.")
            except requests.exceptions.RequestException as e:
                print(f"[FAIL] {ticker}: {filename} -> {e}")

    def download_for_multiple_tickers(self, tickers):
        """Download filings for multiple tickers."""
        for ticker in tickers:
            print(f"\n=== Processing ticker: {ticker} ===")
            self.download_for_ticker(ticker)


# ====== Example Usage ======
if __name__ == "__main__":
    downloader = SECFilingsDownloader(user_agent="Prakhar Agarwal prakhar@example.com")

    tickers = ["AAPL", "AMZN", "MSFT"]  # Add as many tickers as needed
    downloader.download_for_multiple_tickers(tickers)



=== Processing ticker: AAPL ===
[OK] AAPL: aapl-20240928.htm downloaded.
[OK] AAPL: aapl-20230930.htm downloaded.
[OK] AAPL: aapl-20220924.htm downloaded.
[OK] AAPL: aapl-20210925.htm downloaded.
[OK] AAPL: aapl-20200926.htm downloaded.
[OK] AAPL: amzn-20241231.htm downloaded.
[OK] AAPL: amzn-20231231.htm downloaded.
[OK] AAPL: amzn-20221231.htm downloaded.
[OK] AAPL: amzn-20211231.htm downloaded.
[OK] AAPL: amzn-20201231.htm downloaded.
[OK] AAPL: msft-20250630.htm downloaded.
[OK] AAPL: msft-20240630.htm downloaded.
[OK] AAPL: msft-20230630.htm downloaded.

=== Processing ticker: AMZN ===
[OK] AMZN: aapl-20240928.htm downloaded.
[OK] AMZN: aapl-20230930.htm downloaded.
[OK] AMZN: aapl-20220924.htm downloaded.
[OK] AMZN: aapl-20210925.htm downloaded.
[OK] AMZN: aapl-20200926.htm downloaded.
[OK] AMZN: amzn-20241231.htm downloaded.
[OK] AMZN: amzn-20231231.htm downloaded.
[OK] AMZN: amzn-20221231.htm downloaded.
[OK] AMZN: amzn-20211231.htm downloaded.
[OK] AMZN: amzn-20201231.htm dow

In [ ]:
import os
import pandas as pd
import requests

class SECFilingsDownloader:
    def __init__(self, base_download_path="download", user_agent="Your Name Contact@domain.com"):
        self.base_download_path = base_download_path
        self.headers = {
            "User-Agent": user_agent,
            "Accept-Encoding": "gzip, deflate",
            "Host": "www.sec.gov"
        }

    def download_for_ticker(self, ticker):
        """Download files for a single ticker from its latest.csv"""
        latest_csv_path = os.path.join(self.base_download_path, "latest", "latest.csv")

        if not os.path.exists(latest_csv_path):
            print(f"⚠ latest.csv not found for {ticker}, skipping.")
            return

        df = pd.read_csv(latest_csv_path)

        # Filter only rows that belong to this ticker (if column exists)
        if "ticker" in df.columns:
            df = df[df["ticker"].str.lower() == ticker.lower()]

        ticker_folder = os.path.join(self.base_download_path, ticker)
        os.makedirs(ticker_folder, exist_ok=True)

        for _, row in df.iterrows():
            url = row['url']
            filename = os.path.basename(url)
            filepath = os.path.join(ticker_folder, filename)

            if os.path.exists(filepath):
                print(f"Skipping {ticker}: {filename} (already exists)")
                continue

            try:
                response = requests.get(url, headers=self.headers)
                response.raise_for_status()
                with open(filepath, "wb") as f:
                    f.write(response.content)
                print(f"✅ Downloaded {ticker}: {filename}")
            except requests.exceptions.RequestException as e:
                print(f"❌ Failed {ticker}: {filename} - {e}")

    def download_for_multiple_tickers(self, tickers):
        """Download filings for multiple tickers"""
        for ticker in tickers:
            print(f"\n📂 Processing {ticker}...")
            self.download_for_ticker(ticker)


if __name__ == "__main__":
    tickers = ["AMZN", "AAPL", "MSFT"]  # Example tickers
    downloader = SECFilingsDownloader(user_agent="Your Name your@email.com")
    downloader.download_for_multiple_tickers(tickers)



📂 Processing AMZN...
⚠ latest.csv not found for AMZN, skipping.

📂 Processing AAPL...
⚠ latest.csv not found for AAPL, skipping.

📂 Processing MSFT...
⚠ latest.csv not found for MSFT, skipping.


In [ ]:
import os
import pandas as pd
import requests

class SECFilingDownloader:
    def __init__(self, latest_csv_path, base_download_path="download", user_agent="Your Name Contact@domain.com"):
        self.latest_csv_path = latest_csv_path
        self.base_download_path = base_download_path
        self.headers = {
            "User-Agent": user_agent,
            "Accept-Encoding": "gzip, deflate",
            "Host": "www.sec.gov"
        }
        self.data = self._load_csv()

    def _load_csv(self):
        """Load latest.csv file containing all tickers."""
        if not os.path.exists(self.latest_csv_path):
            raise FileNotFoundError(f"File not found: {self.latest_csv_path}")
        return pd.read_csv(self.latest_csv_path)

    def _download_file(self, url, filepath):
        """Download a file if not already present."""
        if os.path.exists(filepath):
            print(f"Skipping {os.path.basename(filepath)} (already exists)")
            return
        try:
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            with open(filepath, "wb") as f:
                f.write(response.content)
            print(f"Downloaded: {os.path.basename(filepath)}")
        except requests.exceptions.RequestException as e:
            print(f"Failed to download {os.path.basename(filepath)}: {e}")

    def download_for_ticker(self, ticker):
        """Download all filings for a specific ticker."""
        ticker_folder = os.path.join(self.base_download_path, ticker)
        os.makedirs(ticker_folder, exist_ok=True)

        ticker_df = self.data[self.data['ticker'].str.upper() == ticker.upper()]
        if ticker_df.empty:
            print(f"No data found for ticker: {ticker}")
            return

        for _, row in ticker_df.iterrows():
            url = row['url']
            filename = os.path.basename(url)
            filepath = os.path.join(ticker_folder, filename)
            self._download_file(url, filepath)

    def download_all(self):
        """Download filings for all tickers in CSV."""
        tickers = self.data['ticker'].str.upper().unique()
        for ticker in tickers:
            print(f"\n=== Downloading files for {ticker} ===")
            self.download_for_ticker(ticker)


if __name__ == "__main__":
    latest_csv_path = "download/file/latest.csv"  # single combined file
    downloader = SECFilingDownloader(latest_csv_path)

    # Option 1: Download for specific tickers
    # downloader.download_for_ticker("AMZN")
    # downloader.download_for_ticker("AAPL")

    # Option 2: Download for all tickers
    downloader.download_all()



=== Downloading files for AAPL ===
Skipping aapl-20240928.htm (already exists)
Skipping aapl-20230930.htm (already exists)
Skipping aapl-20220924.htm (already exists)
Skipping aapl-20210925.htm (already exists)
Skipping aapl-20200926.htm (already exists)

=== Downloading files for AMZN ===
Skipping amzn-20241231.htm (already exists)
Skipping amzn-20231231.htm (already exists)
Skipping amzn-20221231.htm (already exists)
Skipping amzn-20211231.htm (already exists)
Skipping amzn-20201231.htm (already exists)

=== Downloading files for MSFT ===
Skipping msft-20250630.htm (already exists)
Skipping msft-20240630.htm (already exists)
Skipping msft-20230630.htm (already exists)


In [ ]:
#! pip install unstructured
! pip install -U langchain langchain-core



^C


In [ ]:
from config_loader import *
from langchain_community.document_loaders import UnstructuredHTMLLoader
tinker="AMZN"
latest_file="amzn-20241231.htm"
loader=UnstructuredHTMLLoader(f"../download/{tinker}/{latest_file}",)

documents = loader.load()

# Print the loaded content
for doc in documents:
    print(doc.page_content)
    print(doc.metadata)

ModuleNotFoundError: No module named 'pydantic._internal._std_types_schema'

In [ ]:
from langchain.text_splitter import S

In [ ]:
! pip install sec_edgar_downloader

In [ ]:
import os
import re
import json
from bs4 import BeautifulSoup
from datetime import datetime

# ====== CONFIG ======
INPUT_FOLDER = "./download/10k_html"       # folder containing downloaded SEC 10-K HTML files
OUTPUT_FOLDER = "./download/10k_json"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ====== HTML CLEANER ======
def clean_html(file_path):
    """Remove headers, footers, tables, and normalize spaces."""
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html_content = f.read()

    soup = BeautifulSoup(html_content, "html.parser")

    # Remove header/footer/tables
    for tag in soup.find_all(["header", "footer", "table"]):
        tag.decompose()

    # Remove div/span with header/footer class or id
    for tag in soup.find_all(["div", "span"], class_=re.compile(r"(header|footer)", re.I)):
        tag.decompose()
    for tag in soup.find_all(["div", "span"], id=re.compile(r"(header|footer)", re.I)):
        tag.decompose()

    text = soup.get_text(" ", strip=True)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    # Remove Table of Contents section
    toc_pattern = re.compile(r"Table of Contents.*?(?=PART\s+I)", re.IGNORECASE | re.DOTALL)
    text = re.sub(toc_pattern, "", text)

    return text.strip()

# ====== SPLIT BY PART ======
def split_parts(text):
    """Splits text into PART sections."""
    parts = re.split(r"(PART\s+[IVX]+)", text, flags=re.IGNORECASE)
    part_dict = {}
    for i in range(1, len(parts), 2):
        part_title = parts[i].strip().upper()
        part_content = parts[i+1].strip()
        part_dict[part_title] = part_content
    return part_dict

# ====== SPLIT PART INTO ITEMS ======
def split_items(part_text):
    """Splits a PART into ITEMs."""
    items = re.split(r"(ITEM\s+\d+[A-Z]?(?:\.\d+)?)", part_text, flags=re.IGNORECASE)
    item_dict = {}
    for i in range(1, len(items), 2):
        item_title = re.sub(r"\s+", " ", items[i].strip())
        item_content = items[i+1].strip()
        item_dict[item_title] = item_content
    return item_dict

# ====== GET FILING DATE ======
def extract_filing_date(filename, text):
    """Try to get filing date from filename or inside text."""
    # From filename (pattern: TICKER-YYYYMMDD.htm)
    match = re.search(r"(\d{4})(\d{2})(\d{2})", filename)
    if match:
        return f"{match.group(1)}-{match.group(2)}-{match.group(3)}"

    # From inside text (look for 'Filed on' pattern)
    match = re.search(r"Filed on\s+([A-Za-z]+\s+\d{1,2},\s+\d{4})", text)
    if match:
        try:
            return datetime.strptime(match.group(1), "%B %d, %Y").strftime("%Y-%m-%d")
        except:
            pass
    return "Unknown"

# ====== PROCESS SINGLE FILE ======
def process_10k(file_path):
    filename = os.path.basename(file_path)
    ticker = filename.split("-")[0].upper() if "-" in filename else "UNKNOWN"

    # Clean text
    full_text = clean_html(file_path)

    # Get filing date
    filing_date = extract_filing_date(filename, full_text)

    # Split into parts & items
    parts = split_parts(full_text)

    final_json = {
        "ticker": ticker,
        "date_of_filing": filing_date
    }
    for part_title, part_content in parts.items():
        final_json[part_title] = split_items(part_content)

    return final_json

# ====== PROCESS ALL FILES ======
if __name__ == "__main__":
    for file in os.listdir(INPUT_FOLDER):
        if file.lower().endswith((".htm", ".html")):
            file_path = os.path.join(INPUT_FOLDER, file)
            print(f"Processing {file}...")
            structured_data = process_10k(file_path)

            output_path = os.path.join(OUTPUT_FOLDER, file.replace(".htm", ".json").replace(".html", ".json"))
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(structured_data, f, indent=2, ensure_ascii=False)

            print(f"✅ Saved structured JSON → {output_path}")


Processing AAPL-20241101.htm...
✅ Saved structured JSON → ./download/10k_json\AAPL-20241101.json


In [ ]:
from bs4 import BeautifulSoup
from bs4 import XMLParsedAsHTMLWarning
import re
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

In [ ]:
with open("AMZN-20250207.htm", "r", encoding="utf-8", errors="ignore") as f:
    html_text = f.read()


In [ ]:
soup = BeautifulSoup(html_text, "lxml")
candidates = []
for tag in soup.find_all():
    txt = tag.get_text(" ", strip=True).upper()
    if "INDEX" in txt:
        anchors = tag.find_all("a", href=re.compile(r"^#"))
        if len(anchors) >= 0:
            candidates.append((tag, len(anchors)))
candidates.sort(key=lambda x: x[1], reverse=True)

In [ ]:
toc_tag = candidates[0][0]
toc_tag

<html xml:lang="en-US" xmlns="http://www.w3.org/1999/xhtml" xmlns:amzn="http://www.amazon.com/20241231" xmlns:country="http://xbrl.sec.gov/country/2024" xmlns:cyd="http://xbrl.sec.gov/cyd/2024" xmlns:dei="http://xbrl.sec.gov/dei/2024" xmlns:ecd="http://xbrl.sec.gov/ecd/2024" xmlns:iso4217="http://www.xbrl.org/2003/iso4217" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:ixt-sec="http://www.sec.gov/inlineXBRL/transformation/2015-08-31" xmlns:link="http://www.xbrl.org/2003/linkbase" xmlns:srt="http://fasb.org/srt/2024" xmlns:us-gaap="http://fasb.org/us-gaap/2024" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"><head><meta content="text/html" http-equiv="Content-Type"/>
<title>amzn-20241231</title></head><body><div style="display:none"><ix:header><ix:hidden><ix:nonnumeric contex

In [ ]:

toc_tag = candidates[0][0]
anchor_texts = [a.get_text(" ", strip=True) for a in toc_tag.find_all("a", href=re.compile(r"^#"))]

unique_list = list(dict.fromkeys(anchor_texts))
noisy_sentences=['Table of Contents']

cleaned_list = []
for item in unique_list:
    if item == "Form 10-K Summary": 
        break
    if not item.isdigit() and item not in noisy_sentences:
        cleaned_list.append(item)
        print (item)

print(cleaned_list)
print(len(cleaned_list))


Business
Risk Factors
Unresolved Staff Comments
Cybersecurity
Properties
Legal Proceedings
Mine Safety Disclosures
Market for the Registrant’s Common Stock, Related Shareholder Matters, and Issuer Purchases of Equity Securities
Reserved
Management’s Discussion and Analysis of Financial Condition and Results of Operations
Quantitative and Qualitative Disclosures About Market Risk
Financial Statements and Supplementary Data
Changes in and Disagreements with Accountants on Accounting and Financial Disclosure
Controls and Procedures
Other Information
Disclosure Regarding Foreign Jurisdictions that Prevent Inspections
Directors, Executive Officers, and Corporate Governance
Executive Compensation
Security Ownership of Certain Beneficial Owners and Management and Related Shareholder Matters
Certain Relationships and Related Transactions, and Director Independence
Principal Accountant Fees and Services
Exhibits, Financial Statement Schedules
['Business', 'Risk Factors', 'Unresolved Staff Comme

In [ ]:
! pip install weasyprint

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 16.7 MB/s  0:00:00

   ----- ---------------------------------- 1/8 [zopfli]
   ---------- ----------------------------- 2/8 [tinyhtml5]
   ---------- ----------------------------- 2/8 [tinyhtml5]
   --------------- ------------------------ 3/8 [tinycss2]
   --------------- ------------------------ 3/8 [tinycss2]
   -------------------- ------------------- 4/8 [Pyphen]
   ------------------------------ --------- 6/8 [cssselect2]
   ----------------------------------- ---- 7/8 [weasyprint]
   ----------------------------------- ---- 7/8 [weasyprint]
   ----------------------------------- ---- 7/8 [weasyprint]
   ----------------------------------- ---- 7/8 [weasyprint]
   ----------------------------------- ---- 7/8 [weasyprint]
   ----------------------------------- ---- 7/8 [weasyprint]
   ----------------------------------- ---- 7/8 [weasyprint]
   -------------

In [ ]:
from weasyprint import HTML

input_path = "AMZN-20250207.htm"
output_path = "AMZN-20250207-t1.pdf"

HTML(input_path).write_pdf(output_path)

print(f"PDF saved at: {output_path}")


PDF saved at: AMZN-20250207-t1.pdf


In [ ]:
import pdfkit

pdfkit.from_url("AMZN-20250207.htm", 'AMZN.pdf')

OSError: No wkhtmltopdf executable found: "b''"
If this file exists please check that this process can read it or you can pass path to it manually in method call, check README. Otherwise please install wkhtmltopdf - https://github.com/JazzCore/python-pdfkit/wiki/Installing-wkhtmltopdf

In [ ]:
! pip install pymupdf


   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   ---- ----------------------------------- 2.1/18.7 MB 11.8 MB/s eta 0:00:02
   ---------- ----------------------------- 5.0/18.7 MB 12.6 MB/s eta 0:00:02
   ------------------ --------------------- 8.7/18.7 MB 14.5 MB/s eta 0:00:01
   --------------------- ------------------ 10.0/18.7 MB 12.7 MB/s eta 0:00:01
   ------------------------- -------------- 12.1/18.7 MB 12.0 MB/s eta 0:00:01
   ------------------------------ --------- 14.4/18.7 MB 11.9 MB/s eta 0:00:01
   ------------------------------------ --- 17.0/18.7 MB 12.1 MB/s eta 0:00:01
   ---------------------------------------  18.6/18.7 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 18.7/18.7 MB 11.5 MB/s  0:00:01


In [ ]:
import fitz  # PyMuPDF
import re
import json

# -------- CONFIG --------
pdf_path = "AMZN-20250207-t1.pdf"          # Input PDF
output_json_path = "pdf_sections_by_items.json"  # Output JSON
# ------------------------

# Step 1: Open PDF and get all text
doc = fitz.open(pdf_path)
page_texts = [doc[i].get_text("text") for i in range(len(doc))]
full_text = "\n".join(page_texts)

# Step 2: Find all headings like "Item 1. Business" or "Item 1A. Risk Factors"
pattern = re.compile(r'(?im)^\s*Item\s+(\d+[A-Za-z0-9]*)\s*\.?\s*(.+)', re.MULTILINE)
matches = list(pattern.finditer(full_text))

# If no matches found, try uppercase "ITEM"
if not matches:
    pattern2 = re.compile(r'(?m)^\s*ITEM\s+(\d+[A-Za-z0-9]*)\s*\.?\s*(.+)')
    matches = list(pattern2.finditer(full_text))

# Step 3: Build list of (heading, start_position)
headings = []
for m in matches:
    num = m.group(1).strip()
    title_text = m.group(2).strip()
    # Remove trailing page numbers or artifacts
    title_text = re.sub(r'\s+\d+\s*$', '', title_text).strip()
    headings.append((f"Item {num}. {title_text}", m.start()))

# Sort by position in document
headings.sort(key=lambda x: x[1])

# Step 4: Extract sections between headings
sections = {}
for i, (heading, start_pos) in enumerate(headings):
    # Determine where this section ends
    end_pos = headings[i+1][1] if i+1 < len(headings) else len(full_text)
    section_text = full_text[start_pos:end_pos].strip()

    # Normalize key by removing "Item X." prefix
    key = re.sub(r'(?i)^\s*Item\s+\d+[A-Za-z0-9]*\s*\.?\s*', '', heading).strip(" .:")
    sections[key] = section_text

# Step 5: Save as JSON
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(sections, f, indent=4, ensure_ascii=False)

print(f"✅ Extracted {len(sections)} sections to {output_json_path}")


✅ Extracted 26 sections to pdf_sections_by_items.json


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
import re
import json

# -------- CONFIG --------
pdf_path = "AMZN-20250207-t1.pdf"          # Input PDF
output_json_path = "pdf_sections_by_items2.json"  # Output JSON
# ------------------------

# Step 1: Load PDF as text using LangChain
loader = PyMuPDFLoader(pdf_path)
docs = loader.load()

# Step 2: Merge all pages into one text block
full_text = "\n".join(doc.page_content for doc in docs)

# Step 3: Find "Item" headings
pattern = re.compile(r'(?im)^\s*Item\s+(\d+[A-Za-z0-9]*)\s*\.?\s*(.+)', re.MULTILINE)
matches = list(pattern.finditer(full_text))

if not matches:
    pattern2 = re.compile(r'(?m)^\s*ITEM\s+(\d+[A-Za-z0-9]*)\s*\.?\s*(.+)')
    matches = list(pattern2.finditer(full_text))

# Step 4: Store headings and positions
headings = []
for m in matches:
    num = m.group(1).strip()
    title_text = m.group(2).strip()
    title_text = re.sub(r'\s+\d+\s*$', '', title_text).strip()
    headings.append((f"Item {num}. {title_text}", m.start()))

headings.sort(key=lambda x: x[1])

# Step 5: Extract sections
sections = {}
for i, (heading, start_pos) in enumerate(headings):
    end_pos = headings[i+1][1] if i+1 < len(headings) else len(full_text)
    section_text = full_text[start_pos:end_pos].strip()
    key = re.sub(r'(?i)^\s*Item\s+\d+[A-Za-z0-9]*\s*\.?\s*', '', heading).strip(" .:")
    sections[key] = section_text

# Step 6: Save JSON
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(sections, f, indent=4, ensure_ascii=False)

print(f"✅ Extracted {len(sections)} sections to {output_json_path}")


✅ Extracted 26 sections to pdf_sections_by_items2.json


In [23]:
from pdfutils import pdfUtils


In [ ]:
from weasyprint import HTML
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path
import re, os, json

os.environ["G_MESSAGES_DEBUG"] = ""


class PdfUtils:
    def __init__(self, download_dir="download"):
        self.download_pdf_dir = Path(download_dir) / "10k_pdf"
        self.download_json_dir = Path(download_dir) / "10k_json"
        self.download_pdf_dir.mkdir(parents=True, exist_ok=True)
        self.download_json_dir.mkdir(parents=True, exist_ok=True)

    def htm_to_pdf(self, html_folder):
        html_folder = Path(html_folder)
        if not html_folder.exists():
            raise FileNotFoundError(f"HTML folder not found: {html_folder}")

        for html_file in html_folder.glob("*.htm"):
            pdf_file = self.download_pdf_dir / (html_file.stem + ".pdf")
            if pdf_file.exists():
                print(f"[SKIP] PDF already exists: {pdf_file}")
                continue
            try:
                HTML(str(html_file)).write_pdf(str(pdf_file))
                print(f"[OK] Converted: {html_file.name} -> {pdf_file.name}")
            except Exception as e:
                print(f"[ERR] Failed to convert {html_file.name}: {e}")

    def pdf_to_json(self, pdf_folder):
        pdf_folder = Path(pdf_folder)
        if not pdf_folder.exists():
            raise FileNotFoundError(f"PDF folder not found: {pdf_folder}")

        for pdf_file in pdf_folder.glob("*.pdf"):
            json_file = self.download_json_dir / (pdf_file.stem + ".json")
            if json_file.exists():
                print(f"[SKIP] JSON already exists: {json_file}")
                continue
            try:
                loader = PyMuPDFLoader(str(pdf_file))
                docs = loader.load()
                full_text = "\n".join(doc.page_content for doc in docs)

                pattern = re.compile(r'(?im)^\s*Item\s+(\d+[A-Za-z0-9]*)\s*\.?\s*(.+)', re.MULTILINE)
                matches = list(pattern.finditer(full_text))
                if not matches:
                    pattern = re.compile(r'(?m)^\s*ITEM\s+(\d+[A-Za-z0-9]*)\s*\.?\s*(.+)')
                    matches = list(pattern.finditer(full_text))

                headings = []
                for m in matches:
                    num = m.group(1).strip()
                    title_text = m.group(2).strip()
                    title_text = re.sub(r'\s+\d+\s*$', '', title_text).strip()
                    headings.append((f"Item {num}. {title_text}", m.start()))

                headings.sort(key=lambda x: x[1])

                sections = {}
                for i, (heading, start_pos) in enumerate(headings):
                    end_pos = headings[i + 1][1] if i + 1 < len(headings) else len(full_text)
                    section_text = full_text[start_pos:end_pos].strip()
                    key = re.sub(r'(?i)^\s*Item\s+\d+[A-Za-z0-9]*\s*\.?\s*', '', heading).strip(" .:")
                    sections[key] = section_text

                with open(json_file, "w", encoding="utf-8") as f:
                    json.dump(sections, f, indent=4, ensure_ascii=False)

                print(f"[OK] JSON created: {json_file}")
            except Exception as e:
                print(f"[ERR] Failed to convert {pdf_file.name}: {e}")


# Example usage:
# utils = PdfUtils(download_dir="download")
# utils.htm_to_pdf("path/to/html_folder")
# utils.pdf_to_json("download/10k_pdf")


In [26]:
from langchain_community.document_loaders import PyMuPDFLoader
import re
import json

# -------- CONFIG --------
pdf_path = "download\\10k_pdf\\MSFT-20250730.pdf"          # Input PDF
output_json_path = "pdf_sections_by_items3.json"  # Output JSON
# ------------------------

# Step 1: Load PDF as text using LangChain

In [ ]:
loader = PyMuPDFLoader(pdf_path)
docs = loader.load()

In [38]:
docs[4].page_content

"PART I\nItem 1\nNote About Forward-Looking Statements\nThis report includes estimates, projections, statements relating to our business plans, objectives, and  \nexpected operating results that are “forward-looking statements” within the meaning of the Private  \nSecurities Litigation Reform Act of 1995, Section 27A of the Securities Act of 1933, and Section 21E of the \nSecurities  Exchange  Act  of  1934.  Forward-looking  statements  may  appear  throughout  this  report,  \nincluding the following sections: “Business” (Part I, Item 1 of this Form 10-K), “Risk Factors” (Part I, Item \n1A of this Form 10-K), and “Management’s Discussion and Analysis of Financial Condition and Results of \nOperations” (Part II, Item 7 of this Form 10-K). These forward-looking statements generally are identified \nby  the  words  “believe,”  “project,”  “expect,”  “anticipate,”  “estimate,”  “intend,”  “strategy,”  “future,”  \n“opportunity,” “plan,” “may,” “should,” “will,” “would,” “will be,” “will 

In [ ]:



# Step 2: Merge all pages into one text block
full_text = "\n".join(doc.page_content for doc in docs)

# Step 3: Find "Item" headings
pattern = re.compile(r'(?im)^\s*Item\s+(\d+[A-Za-z0-9]*)\s*\.?\s*(.+)', re.MULTILINE)
matches = list(pattern.finditer(full_text))

if not matches:
    pattern2 = re.compile(r'(?m)^\s*ITEM\s+(\d+[A-Za-z0-9]*)\s*\.?\s*(.+)')
    matches = list(pattern2.finditer(full_text))

# Step 4: Store headings and positions
headings = []
for m in matches:
    num = m.group(1).strip()
    title_text = m.group(2).strip()
    title_text = re.sub(r'\s+\d+\s*$', '', title_text).strip()
    headings.append((f"Item {num}. {title_text}", m.start()))

headings.sort(key=lambda x: x[1])

# Step 5: Extract sections
sections = {}
for i, (heading, start_pos) in enumerate(headings):
    end_pos = headings[i+1][1] if i+1 < len(headings) else len(full_text)
    section_text = full_text[start_pos:end_pos].strip()
    key = re.sub(r'(?i)^\s*Item\s+\d+[A-Za-z0-9]*\s*\.?\s*', '', heading).strip(" .:")
    sections[key] = section_text

# Step 6: Save JSON
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(sections, f, indent=4, ensure_ascii=False)

print(f"✅ Extracted {len(sections)} sections to {output_json_path}")
